# 02 — Xử lý mất cân bằng dữ liệu (MILK10k)

**Stage 2.** Backbone tốt nhất từ notebook 01 × **9 phương pháp** × 3 nhánh ảnh, val trên **fold 0**.

| Overlay | Tag | Phương pháp |
|---|---|---|
| `ce_plain` | ce_plain | CE, không bù mất cân bằng (đối chứng) |
| `ce_sqrt_inv` | ce_sqrtinv | CE + trọng số √(1/n) — **trùng stage 1, tự bỏ qua nếu khôi phục output 01** |
| `ce_inv` | ce_inv | CE + trọng số 1/n |
| `cb_effective_num` | cb_b0999 | Class-Balanced loss, β = 0.999 (Cui et al., 2019) |
| `cb_focal` | cb_focal | Class-Balanced Focal, γ = 2 |
| `logit_adjusted` | la_t1 | Logit-adjusted loss, τ = 1 (Menon et al., 2021) |
| `sampler_q05` | samp_q05 | Oversampling P(c) ∝ n_c^0.5 |
| `sampler_q1` | samp_q1 | Oversampling cân bằng hoàn toàn |
| `crt` | crt | Decoupling: train tự nhiên → đóng băng backbone → train lại head với sampler cân bằng (Kang et al., 2020) |

Với mỗi phương pháp: **clinical** → **dermoscopy** → **clinical + dermoscopy**. Một backbone = 27 run (24 run mới).

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. Ô *Cấu hình* đã có sẵn link repo và link dữ liệu; chỉ cần sửa `RESTORE` khi chạy tiếp.
3. **Save Version → Save & Run All (Commit)**: notebook chạy nền tối đa 12 giờ, không cần mở trình duyệt.
   Toàn bộ `outputs/` (log, bảng, file nộp) được lưu vào output của version.
4. **Chưa chạy hết lưới** (hết 12 giờ): Add Input → output của version vừa chạy, điền vào `RESTORE`,
   rồi Save Version lần nữa. Các run đã xong được bỏ qua, run dở dang được chạy lại.

Hai GPU được dùng song song: mỗi GPU chạy một run độc lập (`scripts/run_grid.py`). Checkpoint `.pt` bị xoá
sau khi sinh file nộp để không vượt giới hạn 20GB output; `test_probs.csv` được giữ lại để ensemble.

**Khuyến nghị:** gắn output của notebook 01 vào `RESTORE` để dùng lại 3 run `ce_sqrtinv` và có bảng so sánh đầy đủ.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []
TIME_BUDGET_H = 11.0
BEST_BACKBONES = ["convnext_base"]   # TODO: chọn từ bảng kết quả notebook 01: resnet152 | convnext_base | vit_base | swin_base | dinov2_base
# Loss / phương pháp imbalance cần chạy (tên file trong configs/imbalance/). Mặc định: tất cả 10.
LOSSES = ["ce_plain", "ce_sqrt_inv", "ce_inv", "cb_effective_num", "cb_focal", "logit_adjusted",
          "sampler_q05", "sampler_q1", "crt", "bce"]
VIEWS = ["clin", "derm", "clin+derm"]

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Áp dụng cho MỌI run của lưới, ưu tiên cao nhất (ghi đè configs/default.yaml, configs/baselines/*, configs/env/kaggle_t4.yaml).
# None = giữ giá trị trong config. Một số tham số khác nhau theo backbone, chỉ đặt khi muốn áp dụng đồng loạt:
#   img_size: ResNet-152 320, ConvNeXt/ViT/Swin 224 (Swin-B window 7 bắt buộc 224)
#   batch_size x accum_steps: đặt 16 x 4 (batch hiệu dụng 64) cho MỌI backbone, kể cả ResNet-152
#   (giống các run stage 2 đã train; None = theo config: 16 x 2, ResNet-152 8 x 4)
#   lr: ResNet-152 3e-4, ConvNeXt-B 1e-4, ViT-B / Swin-B 5e-5, DINOv2-B 1e-4
# Lưu ý notebook 06: tham số đặt ở đây (khác None) GHI ĐÈ cả các run ablation R1–R5 -> để None khi chạy ablation.
# Xem bảng "Cấu hình thực tế" ở bước Danh sách run để kiểm tra trước khi train.
PARAMS = {
    # --- Huấn luyện ---
    "train.epochs": 20,                # số epoch tối đa
    "train.patience": 5,               # dừng sớm sau N epoch Dice val không tăng
    "train.warmup_epochs": 1,          # warmup tuyến tính rồi cosine
    "train.lr": None,                  # lr của head / phần mới; None = theo backbone (xem ghi chú phía trên)
    "train.backbone_lr_mult": None,    # lr backbone = lr x hệ số. None = theo config (ViT/Swin 0.2, CNN 0.1);
                                       #   1.0 = fine-tune backbone mạnh như R4 (notebook 06)
    "train.weight_decay": None,
    "train.batch_size": 16,            # số mẫu mỗi lần forward (giới hạn bởi VRAM)
    "train.accum_steps": 4,            # cộng dồn gradient -> batch hiệu dụng = 16 x 4 = 64
    "train.amp": True,                 # mixed precision fp16
    "train.monitor": "dice",           # metric chọn checkpoint: dice | auc | ap | balanced_acc
    "train.folds": [0],                # [0, 1, 2, 3, 4] khi đã chốt method và chạy đủ 5 fold
    "train.full_data": False,          # True: train trên TOÀN BỘ dữ liệu, không val, dùng epoch cuối -> đặt
                                       #   train.epochs = epoch tốt nhất đã thấy khi có val; nên đặt RUN_SUFFIX = "full"
    "seed": 42,
    # --- Dữ liệu & mô hình ---
    "data.img_size": None,             # None = theo backbone (224; ResNet-152 320). DINOv2: bội số của 14 (224, 336, 448)
    "data.resize": None,               # None = "pad" (giữ tỉ lệ + viền đen) | "squash" (resize thẳng, dùng 100% điểm ảnh — R2)
    "data.aug": None,                  # None = "basic" | "strong" (RandomResizedCrop, xoay, màu, blur/nhiễu, xoá vùng — R3)
    "model.pretrained": True,
    "model.dropout": None,             # dropout trước lớp phân loại
    "model.drop_path_rate": None,      # None = 0.0 | 0.1–0.2 giảm overfit backbone (R4)
    "model.mlp_hidden": None,          # None = 512: lớp ẩn MLP 2 lớp sau khi nối 2 ảnh (mô hình ảnh, DINOv2)
    # --- Đánh giá & file nộp ---
    "predict.tta": True,               # trung bình ảnh gốc + lật ngang + lật dọc
    "predict.postprocess": "auto",     # auto: sigmoid nếu loss=bce, ngược lại top1 | top1 | softmax | sigmoid
}
# --- Tham số của từng phương pháp imbalance (chỉ có tác dụng với overlay tương ứng) ---
PARAMS.update({
    "loss.cb_beta": None,              # cb_effective_num, cb_focal (mặc định 0.999)
    "loss.focal_gamma": None,          # cb_focal (mặc định 2.0)
    "loss.la_tau": None,               # logit_adjusted (mặc định 1.0)
    "crt.epochs": None,                # crt: số epoch train lại head (mặc định 10)
    "crt.lr": None,                    # crt: lr của head (mặc định 1e-3)
})
# KHÔNG đặt loss.name / loss.class_weight / train.sampler_q / crt.enabled ở đây: chúng ghi đè cả 9 overlay.

# Đổi tham số so với lần chạy trước -> đặt hậu tố, vd "ep30". Run trùng tên đã xong sẽ bị BỎ QUA,
# nên không có hậu tố thì kết quả cũ được giữ nguyên thay vì train lại.
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (không cần GPU, ~1 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Danh sách run và cấu hình thực tế

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
backbones_arg = f"--backbones {','.join(BEST_BACKBONES)} --overlays {','.join(LOSSES)} --views {','.join(VIEWS)}"
!python scripts/run_grid.py experiments/stage2_imbalance.yaml --env configs/env/kaggle_t4.yaml {backbones_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py experiments/stage2_imbalance.yaml --env configs/env/kaggle_t4.yaml {backbones_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả

In [ ]:
!python scripts/summarize.py --dirs outputs > /dev/null
display(Markdown(open("outputs/summary/summary.md", encoding="utf-8").read()))

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"02_imbalance_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))